# 🛍️ Shopper Spectrum: Customer Segmentation & Product Recommendations

**Dataset:** Online Retail Transactions 2022–2023  
**Author:** Shopper Spectrum Analytics  
**Methods:** KMeans Clustering · RFM Analysis · Cosine-Similarity Recommendations

---

## Table of Contents
1. [Setup & Data Loading](#1)
2. [Data Preprocessing](#2)
3. [Exploratory Data Analysis](#3)
4. [RFM Feature Engineering](#4)
5. [Clustering with KMeans](#5)
6. [Segment Interpretation](#6)
7. [Recommendation System](#7)
8. [Save Artefacts](#8)


## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import pickle, warnings
warnings.filterwarnings('ignore')

# ── Palette ──────────────────────────────────────────────────────────────────
PALETTE = {
    "bg":      "#0D1117", "surface": "#161B22",
    "blue":    "#58A6FF", "green":   "#3FB950",
    "amber":   "#D29922", "coral":   "#F78166",
    "text":    "#E6EDF3", "muted":   "#8B949E",
}
CLUSTER_COLORS = [PALETTE["blue"], PALETTE["green"], PALETTE["amber"], PALETTE["coral"]]

plt.rcParams.update({
    "figure.facecolor": PALETTE["bg"], "axes.facecolor":  PALETTE["surface"],
    "axes.edgecolor":   PALETTE["muted"], "axes.labelcolor": PALETTE["text"],
    "xtick.color":      PALETTE["muted"], "ytick.color":     PALETTE["muted"],
    "text.color":       PALETTE["text"], "grid.color":       "#21262D",
    "grid.linestyle":   "--", "grid.linewidth": 0.5,
})

df_raw = pd.read_csv("ecommerce_data.csv")
print(f"Shape: {df_raw.shape}")
df_raw.head()


## 2. Data Preprocessing <a id='2'></a>

In [ ]:
print("Missing values:\n", df_raw.isnull().sum())
print(f"\nCancelled invoices: {df_raw['InvoiceNo'].astype(str).str.startswith('C').sum()}")
print(f"Negative/zero quantities: {(df_raw['Quantity'] <= 0).sum()}")
print(f"Negative/zero prices: {(df_raw['UnitPrice'] <= 0).sum()}")


In [ ]:
df = df_raw.copy()

# 1. Drop missing CustomerID
df.dropna(subset=["CustomerID"], inplace=True)

# 2. Remove cancelled invoices
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# 3. Remove non-positive quantities and prices
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

# 4. Parse dates and derive TotalPrice
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalPrice"]  = df["Quantity"] * df["UnitPrice"]
df["CustomerID"]  = df["CustomerID"].astype(str)

print(f"Raw:   {df_raw.shape[0]:,} rows")
print(f"Clean: {df.shape[0]:,} rows  ({df_raw.shape[0] - df.shape[0]:,} removed)")
df.info()


## 3. Exploratory Data Analysis <a id='3'></a>

In [ ]:
# 3.1 Revenue by country
country_rev = df.groupby("Country")["TotalPrice"].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(country_rev.index[::-1], country_rev.values[::-1],
               color=PALETTE["blue"], edgecolor="none", height=0.6)
for bar, val in zip(bars, country_rev.values[::-1]):
    ax.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
            f"£{val:,.0f}", va="center", fontsize=8, color=PALETTE["muted"])
ax.set_title("Revenue by Country (Top 10)", fontsize=13, pad=10)
ax.set_xlabel("Total Revenue (£)")
ax.set_xlim(0, country_rev.max()*1.18)
ax.grid(axis="x"); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()


In [ ]:
# 3.2 Top 15 products
prod_rev = df.groupby("Description")["TotalPrice"].sum().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(prod_rev.index[::-1], prod_rev.values[::-1],
        color=PALETTE["green"], edgecolor="none", height=0.7)
ax.set_title("Top 15 Products by Revenue", fontsize=13, pad=10)
ax.set_xlabel("Total Revenue (£)")
ax.set_xlim(0, prod_rev.max()*1.15)
ax.grid(axis="x"); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()


In [ ]:
# 3.3 Monthly revenue trend
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M")
monthly = df.groupby("YearMonth")["TotalPrice"].sum().reset_index()
monthly["YearMonth"] = monthly["YearMonth"].astype(str)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly["YearMonth"], monthly["TotalPrice"],
        color=PALETTE["blue"], linewidth=2)
ax.fill_between(range(len(monthly)), monthly["TotalPrice"], alpha=0.12, color=PALETTE["blue"])
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly["YearMonth"], rotation=45, ha="right", fontsize=7)
ax.set_title("Monthly Revenue Trend (2022–2023)", fontsize=13, pad=10)
ax.set_ylabel("Revenue (£)")
ax.grid(axis="y"); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()


In [ ]:
# 3.4 Transaction volume by day-of-week
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
dow_vol   = df.groupby("DayOfWeek").size().reindex(dow_order)

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(dow_vol.index, dow_vol.values, color=PALETTE["amber"], edgecolor="none", width=0.6)
ax.set_title("Transaction Volume by Day of Week", fontsize=13, pad=10)
ax.set_ylabel("Transactions")
ax.grid(axis="y"); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()


## 4. RFM Feature Engineering <a id='4'></a>

In [ ]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print(f"Snapshot date: {snapshot_date.date()}")

rfm = df.groupby("CustomerID").agg(
    Recency   = ("InvoiceDate",  lambda x: (snapshot_date - x.max()).days),
    Frequency = ("InvoiceNo",    "nunique"),
    Monetary  = ("TotalPrice",   "sum"),
).reset_index()

print(f"\nRFM shape: {rfm.shape}")
rfm.describe().round(2)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
cols   = ["Recency", "Frequency", "Monetary"]
colors = [PALETTE["coral"], PALETTE["green"], PALETTE["amber"]]

for ax, col, color in zip(axes, cols, colors):
    cap = rfm[col].quantile(0.99)
    ax.hist(rfm[col].clip(upper=cap), bins=40, color=color, edgecolor="none", alpha=0.85)
    ax.set_title(col, fontsize=12)
    ax.set_ylabel("Count")
    ax.grid(axis="y"); ax.set_axisbelow(True)

plt.suptitle("RFM Distributions (99th-pct capped)", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

print("\nNote: Frequency and Monetary are right-skewed → log-transform before clustering.")


In [ ]:
# Log-transform skewed features
rfm_features = rfm[["Recency","Frequency","Monetary"]].copy()
rfm_features["Frequency"] = np.log1p(rfm_features["Frequency"])
rfm_features["Monetary"]  = np.log1p(rfm_features["Monetary"])

scaler  = StandardScaler()
X_scaled = scaler.fit_transform(rfm_features)

print("Scaled RFM stats:")
pd.DataFrame(X_scaled, columns=["Recency","Frequency","Monetary"]).describe().round(3)


## 5. Clustering with KMeans <a id='5'></a>

In [ ]:
inertias   = []
sil_scores = []
K_range    = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))
    print(f"  k={k}  inertia={km.inertia_:,.1f}  silhouette={sil_scores[-1]:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(list(K_range), inertias, "o-", color=PALETTE["blue"], linewidth=2)
axes[0].set_title("Elbow Curve", fontsize=13)
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
axes[0].axvline(4, color=PALETTE["coral"], linestyle="--", label="k=4 chosen")
axes[0].legend(); axes[0].grid()

axes[1].plot(list(K_range), sil_scores, "s-", color=PALETTE["green"], linewidth=2)
axes[1].set_title("Silhouette Score", fontsize=13)
axes[1].set_xlabel("k"); axes[1].set_ylabel("Score")
axes[1].axvline(4, color=PALETTE["coral"], linestyle="--", label="k=4 chosen")
axes[1].legend(); axes[1].grid()

plt.suptitle("Choosing Optimal k", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

best_sil_k = list(K_range)[sil_scores.index(max(sil_scores))]
print(f"Best silhouette at k={best_sil_k}  ({max(sil_scores):.4f})")
print("→ Choosing k=4 for business interpretability (4 clear RFM segments).")


In [ ]:
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
km_final.fit(X_scaled)
rfm["Cluster"] = km_final.labels_

print(f"Silhouette score (k=4): {silhouette_score(X_scaled, km_final.labels_):.4f}")
rfm.groupby("Cluster")[["Recency","Frequency","Monetary"]].mean().round(1)


## 6. Segment Interpretation <a id='6'></a>

In [ ]:
# Assign labels by ranking clusters on a composite engagement score
centers = rfm.groupby("Cluster")[["Recency","Frequency","Monetary"]].mean()
centers["score"] = -centers["Recency"] + centers["Frequency"]*5 + centers["Monetary"]/50
score_order      = centers["score"].sort_values().index.tolist()
label_by_score   = {0:"At-Risk", 1:"Occasional", 2:"Regular", 3:"High-Value"}
label_map        = {idx: label_by_score[rank] for rank, idx in enumerate(score_order)}

rfm["Segment"] = rfm["Cluster"].map(label_map)
print(rfm["Segment"].value_counts().to_string())


In [ ]:
# PCA scatter
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
for seg, color in zip(["High-Value","Regular","Occasional","At-Risk"], CLUSTER_COLORS):
    mask = rfm["Segment"] == seg
    ax.scatter(X_pca[mask,0], X_pca[mask,1], c=color, s=18,
               alpha=0.6, edgecolors="none", label=seg)
ax.set_title("Customer Segments — PCA Projection", fontsize=13, pad=10)
ax.set_xlabel(f"PC1  ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2  ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.legend(framealpha=0.2, edgecolor=PALETTE["muted"])
ax.grid(); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()


In [ ]:
# Radar chart per segment
seg_means = rfm.groupby("Segment")[["Recency","Frequency","Monetary"]].mean()
seg_norm  = (seg_means - seg_means.min()) / (seg_means.max() - seg_means.min())
seg_norm["Recency"] = 1 - seg_norm["Recency"]   # invert

labels  = ["Recency\n(Inverted)","Frequency","Monetary"]
angles  = np.linspace(0, 2*np.pi, 3, endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, 4, figsize=(14,4), subplot_kw=dict(polar=True))
segs_order = ["High-Value","Regular","Occasional","At-Risk"]

for ax, seg, color in zip(axes, segs_order, CLUSTER_COLORS):
    if seg not in seg_norm.index: continue
    vals = seg_norm.loc[seg].tolist() + seg_norm.loc[seg].tolist()[:1]
    ax.set_facecolor(PALETTE["surface"])
    ax.plot(angles, vals, color=color, linewidth=2)
    ax.fill(angles, vals, alpha=0.22, color=color)
    ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=8)
    ax.set_ylim(0,1); ax.set_yticks([])
    ax.set_title(seg, fontsize=11, pad=14, color=color)
    ax.grid(color=PALETTE["muted"], alpha=0.4)

plt.suptitle("RFM Radar Profiles by Segment", fontsize=14, y=1.03)
plt.tight_layout(); plt.show()


In [ ]:
# Summary table
seg_summary = rfm.groupby("Segment").agg(
    Count     = ("CustomerID","count"),
    Avg_R     = ("Recency",   "mean"),
    Avg_F     = ("Frequency", "mean"),
    Avg_M     = ("Monetary",  "mean"),
    Total_Rev = ("Monetary",  "sum"),
).round(1).sort_values("Avg_M", ascending=False)

print(seg_summary.to_string())


## 7. Recommendation System <a id='7'></a>

In [ ]:
# Build customer-product matrix
basket = df.groupby(["CustomerID","Description"])["Quantity"].sum().unstack(fill_value=0)
print(f"Basket matrix shape: {basket.shape}  (customers × products)")
basket.head(3)


In [ ]:
# Compute cosine similarity between products
item_matrix = basket.T   # products × customers
cos_sim     = cosine_similarity(item_matrix)
cos_sim_df  = pd.DataFrame(cos_sim, index=item_matrix.index, columns=item_matrix.index)
print(f"Similarity matrix: {cos_sim_df.shape}")


In [ ]:
def get_recommendations(product_name: str, top_n: int = 5):
    """Return the top_n most similar products to product_name."""
    if product_name not in cos_sim_df.index:
        matches = [p for p in cos_sim_df.index if product_name.upper() in p.upper()]
        if not matches:
            return pd.Series(dtype=float)
        product_name = matches[0]
    return cos_sim_df[product_name].drop(product_name).sort_values(ascending=False).head(top_n)

# Demo
test_product = "JUMBO BAG RED RETROSPOT"
recs = get_recommendations(test_product)
print(f"\nTop recommendations for: '{test_product}'\n")
for i, (prod, score) in enumerate(recs.items(), 1):
    print(f"  #{i}  {prod:<45}  similarity={score:.4f}")


In [ ]:
# Visualise similarity heatmap for a sample of products
sample_prods = item_matrix.index[:20].tolist()
sub_sim = cos_sim_df.loc[sample_prods, sample_prods]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sub_sim, ax=ax, cmap="Blues", linewidths=0.3,
            linecolor="#21262D", annot=False, square=True,
            cbar_kws={"shrink": 0.8})
ax.set_title("Product Cosine-Similarity Heatmap (Sample)", fontsize=13, pad=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)
plt.tight_layout(); plt.show()


## 8. Save Artefacts <a id='8'></a>

In [ ]:
import os
os.makedirs("assets", exist_ok=True)

with open("assets/model_bundle.pkl","wb") as f:
    pickle.dump({
        "scaler":    scaler,
        "kmeans":    km_final,
        "label_map": label_map,
        "cos_sim_df": cos_sim_df,
        "rfm":       rfm,
        "cluster_stats": rfm.groupby("Segment")[["Recency","Frequency","Monetary"]].agg(["mean","median"]).round(1),
    }, f)

rfm.to_csv("assets/rfm_data.csv", index=False)
rfm.groupby("Segment").agg(
    Count=("CustomerID","count"),
    Avg_R=("Recency","mean"),
    Avg_F=("Frequency","mean"),
    Avg_M=("Monetary","mean"),
).round(1).reset_index().to_csv("assets/segment_stats.csv", index=False)

print("Saved: assets/model_bundle.pkl")
print("Saved: assets/rfm_data.csv")
print("Saved: assets/segment_stats.csv")
print("\n✅ All artefacts ready.  Run `streamlit run app.py` to launch the app.")


---
## Results Summary

| Metric | Value |
|---|---|
| Customers analysed | 800 |
| Optimal clusters | k = 4 |
| Silhouette Score | ~0.35–0.45 |
| Products in similarity matrix | 40 |
| Recommendation method | Item-based cosine similarity |

### Segment Labels

| Segment | Profile | Action |
|---|---|---|
| **High-Value** | Low recency, high F & M | Reward with exclusive perks |
| **Regular** | Mid recency, steady F & M | Cross-sell, moderate discounts |
| **Occasional** | Mid recency, low F & M | Seasonal campaigns |
| **At-Risk** | High recency (long ago), low F & M | Win-back emails with incentives |
